# NeuroGolf Solver Family: Fill / Additive Marking - nonlocal_1color v15tmp

Strict task002 validation notebook. Unlike v14tmp, this notebook requires the real file `Co_Kaggle/g3/competition_material/taskfiles/task002.json`; it does not fall back to embedded smoke examples. It builds a compact static CNN-style ONNX reachability/flood-fill model, validates it on `train + test + arc-gen`, prints profiling, and creates a task002-only submission zip.

In [ ]:
import json, zipfile
from collections import Counter, deque
from pathlib import Path
import numpy as np
try:
    import pandas as pd
except Exception:
    pd = None
try:
    import onnx, onnxruntime as ort
    from onnx import TensorProto, helper, numpy_helper
except Exception as exc:
    onnx = ort = helper = numpy_helper = TensorProto = None
    print('ONNX imports unavailable:', repr(exc))

BATCH, CH, H, W = 1, 10, 30, 30
TASK_ID = 'task002'
MODEL_VERSION = 'fill-additive-nonlocal-1color-v0.15tmp-task002-cnn-reachability-strict'
PROPAGATION_STEPS = 64
MAX_ONNX_FILE_BYTES = 1_440_000
TASK_JSON = Path('Co_Kaggle/g3/competition_material/taskfiles/task002.json')
OUT_DIR = Path('working_submission/fill_enclosed_regions_nonlocal_1color_v15tmp')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('MODEL_VERSION:', MODEL_VERSION)
print('TASK_JSON:', TASK_JSON.resolve())
print('OUT_DIR:', OUT_DIR.resolve())
assert TASK_JSON.exists(), f'Missing required task file: {TASK_JSON}'

In [ ]:
def load_task(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)

def grid_shape(g):
    return len(g), len(g[0]) if g else 0

def grid_to_tensor(grid):
    arr = np.zeros((BATCH, CH, H, W), dtype=np.float32)
    for r,row in enumerate(grid):
        for c,color in enumerate(row):
            arr[0, int(color), r, c] = 1.0
    return arr

def tensor_to_grid(arr, h, w):
    arr = np.asarray(arr)[0]
    out = []
    for r in range(h):
        row = []
        for c in range(w):
            vals = np.where(arr[:, r, c] > 0.5)[0]
            row.append(int(vals[0]) if len(vals) == 1 else -9)
        out.append(row)
    return out

def simulate_task002_floodfill(inp):
    h,w = grid_shape(inp)
    pred = [list(row) for row in inp]
    def bg(r,c): return 0 <= r < h and 0 <= c < w and int(inp[r][c]) == 0
    q, seen = deque(), set()
    for r in range(h):
        for c in range(w):
            if (r in (0,h-1) or c in (0,w-1)) and bg(r,c):
                q.append((r,c)); seen.add((r,c))
    while q:
        r,c = q.popleft()
        for dr,dc in ((1,0),(-1,0),(0,1),(0,-1)):
            nr,nc = r+dr, c+dc
            if bg(nr,nc) and (nr,nc) not in seen:
                seen.add((nr,nc)); q.append((nr,nc))
    for r in range(h):
        for c in range(w):
            if int(inp[r][c]) == 0 and (r,c) not in seen:
                pred[r][c] = 4
    return pred

def eval_grid(task, fn):
    rows=[]; right=total=0; first_wrong=None
    for split in ['train','test','arc-gen']:
        sr=st=0
        for i,ex in enumerate(task.get(split, [])):
            ok = fn(ex['input']) == ex['output']
            sr += int(ok); st += 1; right += int(ok); total += 1
            if not ok and first_wrong is None:
                first_wrong = f'{split}[{i}]'
        rows.append({'split':split,'right':sr,'total':st,'accuracy':sr/st if st else None})
    return {'right':right,'total':total,'accuracy':right/total if total else None,'first_wrong':first_wrong,'rows':rows}

task = load_task(TASK_JSON)
print('loaded task002 examples:', {k: len(task.get(k, [])) for k in ['train','test','arc-gen']})
print('shape modes:', Counter(f"{len(ex['input'])}x{len(ex['input'][0])}" for split in ['train','test','arc-gen'] for ex in task.get(split, [])).most_common(10))
ref_summary = eval_grid(task, simulate_task002_floodfill)
print('python reference accuracy:', ref_summary['right'], '/', ref_summary['total'], ref_summary['accuracy'], 'first_wrong=', ref_summary['first_wrong'])
assert ref_summary['right'] == ref_summary['total'], 'Python flood-fill reference does not match task002 visible examples'
if pd is not None: display(pd.DataFrame(ref_summary['rows']))

In [ ]:
def require_onnx():
    if onnx is None or helper is None or TensorProto is None or numpy_helper is None or ort is None:
        raise ImportError('onnx and onnxruntime are required for export/validation')

def init(name, arr):
    return numpy_helper.from_array(np.asarray(arr, dtype=np.float32), name=name)

def make_task002_cnn_model(n_steps=PROPAGATION_STEPS):
    require_onnx()
    inp = helper.make_tensor_value_info('input', TensorProto.FLOAT, [BATCH, CH, H, W])
    out = helper.make_tensor_value_info('output', TensorProto.FLOAT, [BATCH, CH, H, W])
    nodes, inits = [], []
    w_bg = np.zeros((1,CH,1,1), np.float32); w_bg[0,0,0,0] = 1
    w_valid = np.ones((1,CH,1,1), np.float32)
    k = np.zeros((1,1,3,3), np.float32)
    k[0,0,1,1]=k[0,0,0,1]=k[0,0,2,1]=k[0,0,1,0]=k[0,0,1,2]=1
    c5 = np.array([[[[5.0]]]], np.float32)
    w_delta = np.zeros((CH,1,1,1), np.float32); w_delta[0,0,0,0] = -1; w_delta[4,0,0,0] = 1
    inits += [init('W_bg',w_bg), init('W_valid',w_valid), init('K_cross',k), init('C5',c5), init('W_delta',w_delta)]
    nodes += [
        helper.make_node('Conv', ['input','W_bg'], ['bg'], kernel_shape=[1,1]),
        helper.make_node('Conv', ['input','W_valid'], ['valid_raw'], kernel_shape=[1,1]),
        helper.make_node('Clip', ['valid_raw'], ['valid'], min=0.0, max=1.0),
        helper.make_node('Conv', ['valid','K_cross'], ['valid_cross'], kernel_shape=[3,3], pads=[1,1,1,1]),
        helper.make_node('Sub', ['C5','valid_cross'], ['edge_raw']),
        helper.make_node('Clip', ['edge_raw'], ['edge'], min=0.0, max=1.0),
        helper.make_node('Mul', ['bg','edge'], ['reach_0']),
    ]
    prev='reach_0'
    for i in range(1, n_steps+1):
        nodes += [
            helper.make_node('Conv', [prev,'K_cross'], [f'reach_neigh_{i}'], kernel_shape=[3,3], pads=[1,1,1,1]),
            helper.make_node('Clip', [f'reach_neigh_{i}'], [f'reach_bin_{i}'], min=0.0, max=1.0),
            helper.make_node('Mul', ['bg',f'reach_bin_{i}'], [f'reach_{i}']),
        ]
        prev=f'reach_{i}'
    nodes += [
        helper.make_node('Sub', ['bg',prev], ['enclosed_raw']),
        helper.make_node('Clip', ['enclosed_raw'], ['enclosed'], min=0.0, max=1.0),
        helper.make_node('Conv', ['enclosed','W_delta'], ['delta'], kernel_shape=[1,1]),
        helper.make_node('Add', ['input','delta'], ['output']),
    ]
    model = helper.make_model(helper.make_graph(nodes, 'task002_cnn_reachability_v15tmp', [inp], [out], inits), ir_version=10, opset_imports=[helper.make_opsetid('',10)])
    onnx.checker.check_model(model)
    return model

def profile_model(model, path):
    params=static_bytes=0
    init_rows=[]
    for x in model.graph.initializer:
        arr = numpy_helper.to_array(x)
        params += int(np.prod(arr.shape)); static_bytes += int(arr.nbytes)
        init_rows.append({'name':x.name,'shape':tuple(arr.shape),'bytes':int(arr.nbytes)})
    return {'task_id':TASK_ID,'model_version':MODEL_VERSION,'file_size_bytes':Path(path).stat().st_size,'params':params,'nodes':len(model.graph.node),'op_counts':dict(sorted(Counter(n.op_type for n in model.graph.node).items())),'static_memory_bytes':static_bytes,'file_size_ok':Path(path).stat().st_size <= MAX_ONNX_FILE_BYTES,'initializers':init_rows}

def validate_onnx(path, task):
    sess = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    rows=[]; right=total=0; first_wrong=None
    for split in ['train','test','arc-gen']:
        sr=st=0
        for i,ex in enumerate(task.get(split, [])):
            pred = sess.run(None, {'input': grid_to_tensor(ex['input'])})[0]
            h,w = grid_shape(ex['output'])
            ok = tensor_to_grid(pred,h,w) == ex['output']
            sr += int(ok); st += 1; right += int(ok); total += 1
            if not ok and first_wrong is None:
                first_wrong = f'{split}[{i}]'
        rows.append({'split':split,'right':sr,'total':st,'accuracy':sr/st if st else None})
    return {'right':right,'total':total,'accuracy':right/total if total else None,'first_wrong':first_wrong,'rows':rows}

In [ ]:
for old in OUT_DIR.glob('task*.onnx'):
    old.unlink()
model = make_task002_cnn_model(PROPAGATION_STEPS)
model_path = OUT_DIR/'task002.onnx'
onnx.save(model, model_path)
profile = profile_model(model, model_path)
print('saved:', model_path)
print('file_size_bytes:', profile['file_size_bytes'])
print('params:', profile['params'])
print('nodes:', profile['nodes'])
print('op_counts:', json.dumps(profile['op_counts'], sort_keys=True))
print('static_memory_bytes:', profile['static_memory_bytes'])
print('file_size_ok:', profile['file_size_ok'])
assert profile['file_size_ok'], 'ONNX file exceeds 1.44 MB'
onnx_summary = validate_onnx(model_path, task)
print('onnx visible accuracy:', onnx_summary['right'], '/', onnx_summary['total'], onnx_summary['accuracy'], 'first_wrong=', onnx_summary['first_wrong'])
assert onnx_summary['right'] == onnx_summary['total'], 'ONNX model failed full visible validation'
if pd is not None:
    display(pd.DataFrame(onnx_summary['rows']))
    display(pd.DataFrame(profile['initializers']))

In [ ]:
zip_path = OUT_DIR/'submission.zip'
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(model_path, arcname='task002.onnx')
print('selected task ids:', [TASK_ID])
print('models saved:', len(list(OUT_DIR.glob('task*.onnx'))))
print('family zip:', zip_path)
with zipfile.ZipFile(zip_path) as zf:
    print('zip members:', [(i.filename, i.file_size) for i in zf.infolist()])
final_row = {**{k:profile[k] for k in ['task_id','model_version','file_size_bytes','params','nodes','op_counts','static_memory_bytes','file_size_ok']}, 'visible_right':onnx_summary['right'], 'visible_total':onnx_summary['total'], 'visible_accuracy':onnx_summary['accuracy'], 'first_wrong':onnx_summary['first_wrong'], 'propagation_steps':PROPAGATION_STEPS, 'export_status':'task002_cnn_reachability_export_strict'}
if pd is not None: display(pd.DataFrame([final_row]))
else: print(json.dumps(final_row, indent=2, sort_keys=True))